# EDA: `train_word_transcripts.jsonl`

This notebook performs an extensive exploratory data analysis (EDA) of competition labels in `data/train_word_transcripts.jsonl`.

It covers:
- dataset shape and schema checks
- data quality and duplicate diagnostics
- label/text distributions
- duration and file-size behavior
- child/session-level coverage
- token, n-gram, and lexical diversity analysis
- outlier surfacing for manual review

In [ ]:
from pathlib import Path
import json
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)

DATA_PATH = Path("data/train_word_transcripts.jsonl")
assert DATA_PATH.exists(), f"Missing file: {DATA_PATH}"
print(f"Using file: {DATA_PATH.resolve()}")

In [ ]:
# Load JSONL
rows = []
with DATA_PATH.open() as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
df.head(3)

In [ ]:
# Schema and type checks
expected_cols = [
    "utterance_id", "child_id", "session_id", "audio_path", "audio_duration_sec",
    "age_bucket", "md5_hash", "filesize_bytes", "orthographic_text"
]

print("Missing expected columns:", sorted(set(expected_cols) - set(df.columns)))
print("Unexpected columns:", sorted(set(df.columns) - set(expected_cols)))
print()
print(df.dtypes)
print()
print(df.describe(include="all").T)

In [ ]:
# Missing values and blank text diagnostics
missing_summary = df.isna().sum().to_frame("missing_count")
missing_summary["missing_pct"] = 100 * missing_summary["missing_count"] / len(df)
missing_summary.sort_values("missing_count", ascending=False)


In [ ]:
blank_text_mask = df["orthographic_text"].astype(str).str.strip().eq("")
print(f"Blank text rows: {blank_text_mask.sum():,} ({100*blank_text_mask.mean():.3f}%)")

non_ascii_mask = ~df["orthographic_text"].fillna("").str.match(r"^[\x00-\x7F]*$")
print(f"Rows with non-ASCII chars in text: {non_ascii_mask.sum():,}")


In [ ]:
# Uniqueness and duplicate checks
id_uniqueness = {
    "utterance_id_nunique": df["utterance_id"].nunique(),
    "child_id_nunique": df["child_id"].nunique(),
    "session_id_nunique": df["session_id"].nunique(),
    "md5_hash_nunique": df["md5_hash"].nunique(),
    "audio_path_nunique": df["audio_path"].nunique(),
}
print(id_uniqueness)

print("\nDuplicate utterance_id rows:", int(df.duplicated(subset=["utterance_id"]).sum()))
print("Duplicate audio_path rows:", int(df.duplicated(subset=["audio_path"]).sum()))
print("Duplicate md5_hash rows:", int(df.duplicated(subset=["md5_hash"]).sum()))
print("Exact duplicate full rows:", int(df.duplicated().sum()))


In [ ]:
# Age bucket distribution
age_counts = df["age_bucket"].value_counts(dropna=False)
age_pct = (100 * age_counts / len(df)).round(2)
age_dist = pd.DataFrame({"count": age_counts, "pct": age_pct})
display(age_dist)

plt.figure(figsize=(8, 4))
sns.barplot(x=age_counts.index, y=age_counts.values, color="#4C78A8")
plt.title("Age Bucket Distribution")
plt.xlabel("age_bucket")
plt.ylabel("count")
plt.tight_layout()
plt.show()

In [ ]:
# Numeric distribution diagnostics
num_cols = ["audio_duration_sec", "filesize_bytes"]

display(df[num_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df["audio_duration_sec"], bins=60, kde=True, ax=axes[0], color="#72B7B2")
axes[0].set_title("Audio Duration Distribution")
axes[0].set_xlabel("seconds")

sns.histplot(df["filesize_bytes"], bins=60, kde=True, ax=axes[1], color="#F58518")
axes[1].set_title("File Size Distribution")
axes[1].set_xlabel("bytes")
plt.tight_layout()
plt.show()

In [ ]:
# Duration-size relationship
plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=df.sample(min(20000, len(df)), random_state=42),
    x="audio_duration_sec", y="filesize_bytes", alpha=0.2, s=20
)
plt.title("File Size vs Audio Duration (sample)")
plt.tight_layout()
plt.show()

corr = df[["audio_duration_sec", "filesize_bytes"]].corr().iloc[0, 1]
print(f"Pearson correlation (duration, filesize): {corr:.4f}")

In [ ]:
# Build text-derived features
text = df["orthographic_text"].fillna("").astype(str)

def tokenize(s: str):
    s = s.lower()
    s = re.sub(r"[^a-z0-9' ]+", " ", s)
    toks = [t for t in s.split() if t]
    return toks

df["char_count"] = text.str.len()
df["word_count"] = text.apply(lambda s: len(tokenize(s)))
df["avg_word_len"] = text.apply(lambda s: np.mean([len(t) for t in tokenize(s)]) if tokenize(s) else 0.0)
df["words_per_second"] = df["word_count"] / df["audio_duration_sec"].replace(0, np.nan)

feature_cols = ["char_count", "word_count", "avg_word_len", "words_per_second"]
display(df[feature_cols].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T)


In [ ]:
# Text feature distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
plot_cols = ["char_count", "word_count", "avg_word_len", "words_per_second"]
colors = ["#4C78A8", "#72B7B2", "#F58518", "#E45756"]

for ax, col, c in zip(axes.ravel(), plot_cols, colors):
    sns.histplot(df[col].replace([np.inf, -np.inf], np.nan).dropna(), bins=60, kde=True, ax=ax, color=c)
    ax.set_title(col)

plt.tight_layout()
plt.show()

In [ ]:
# Top tokens and lexical diversity
all_tokens = []
for s in text:
    all_tokens.extend(tokenize(s))

token_counts = Counter(all_tokens)
print(f"Total tokens: {len(all_tokens):,}")
print(f"Unique tokens (vocab size): {len(token_counts):,}")
print(f"Type-token ratio: {len(token_counts)/max(len(all_tokens),1):.5f}")

top_tokens = pd.DataFrame(token_counts.most_common(50), columns=["token", "count"])
display(top_tokens.head(20))

plt.figure(figsize=(10, 8))
sns.barplot(data=top_tokens.head(20), y="token", x="count", color="#54A24B")
plt.title("Top 20 Tokens")
plt.tight_layout()
plt.show()

In [ ]:
# Top bigrams
bigram_counts = Counter()
for s in text:
    toks = tokenize(s)
    bigram_counts.update(zip(toks, toks[1:]))

top_bigrams = pd.DataFrame(
    [(f"{a} {b}", c) for (a, b), c in bigram_counts.most_common(30)],
    columns=["bigram", "count"]
)
display(top_bigrams.head(20))

plt.figure(figsize=(10, 8))
sns.barplot(data=top_bigrams.head(20), y="bigram", x="count", color="#B279A2")
plt.title("Top 20 Bigrams")
plt.tight_layout()
plt.show()

In [ ]:
# Per-child and per-session coverage
child_stats = df.groupby("child_id").agg(
    n_utterances=("utterance_id", "count"),
    total_duration_sec=("audio_duration_sec", "sum"),
    mean_words=("word_count", "mean")
).sort_values("n_utterances", ascending=False)

session_stats = df.groupby("session_id").agg(
    n_utterances=("utterance_id", "count"),
    total_duration_sec=("audio_duration_sec", "sum"),
    n_children=("child_id", "nunique")
).sort_values("n_utterances", ascending=False)

print("Child-level summary")
display(child_stats.describe(percentiles=[0.5, 0.9, 0.99]).T)
print("Session-level summary")
display(session_stats.describe(percentiles=[0.5, 0.9, 0.99]).T)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(child_stats["n_utterances"], bins=60, ax=axes[0], color="#4C78A8")
axes[0].set_title("Utterances per Child")

sns.histplot(session_stats["n_utterances"], bins=60, ax=axes[1], color="#F58518")
axes[1].set_title("Utterances per Session")
plt.tight_layout()
plt.show()

In [ ]:
# Age bucket comparison of speaking rate and duration
agg = df.groupby("age_bucket").agg(
    n=("utterance_id", "count"),
    duration_mean=("audio_duration_sec", "mean"),
    words_mean=("word_count", "mean"),
    wps_mean=("words_per_second", "mean")
).sort_values("n", ascending=False)

display(agg)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(data=df, x="age_bucket", y="audio_duration_sec", ax=axes[0], color="#72B7B2")
axes[0].set_title("Duration by Age Bucket")

sns.boxplot(data=df.replace([np.inf, -np.inf], np.nan), x="age_bucket", y="words_per_second", ax=axes[1], color="#E45756")
axes[1].set_title("Words/sec by Age Bucket")
plt.tight_layout()
plt.show()

In [ ]:
# Outlier surfacing for manual inspection
# Define conservative outliers using quantile thresholds
q = df[["audio_duration_sec", "word_count", "words_per_second", "filesize_bytes"]].quantile([0.01, 0.99])
q01, q99 = q.loc[0.01], q.loc[0.99]

outlier_mask = (
    (df["audio_duration_sec"] < q01["audio_duration_sec"]) |
    (df["audio_duration_sec"] > q99["audio_duration_sec"]) |
    (df["word_count"] < q01["word_count"]) |
    (df["word_count"] > q99["word_count"]) |
    (df["words_per_second"] < q01["words_per_second"]) |
    (df["words_per_second"] > q99["words_per_second"]) |
    (df["filesize_bytes"] < q01["filesize_bytes"]) |
    (df["filesize_bytes"] > q99["filesize_bytes"])
)

outliers = df.loc[outlier_mask, [
    "utterance_id", "child_id", "session_id", "age_bucket",
    "audio_duration_sec", "filesize_bytes", "word_count", "words_per_second", "orthographic_text"
]].copy()

print(f"Outlier candidates: {len(outliers):,} ({100*len(outliers)/len(df):.2f}%)")
outliers.head(20)


In [ ]:
# Optional: save key summary artifacts for downstream modeling notes
summary_dir = Path("statisitcs")
summary_dir.mkdir(parents=True, exist_ok=True)

age_dist.to_csv(summary_dir / "eda_age_distribution.csv")
child_stats.reset_index().to_csv(summary_dir / "eda_child_stats.csv", index=False)
session_stats.reset_index().to_csv(summary_dir / "eda_session_stats.csv", index=False)
top_tokens.to_csv(summary_dir / "eda_top_tokens.csv", index=False)
top_bigrams.to_csv(summary_dir / "eda_top_bigrams.csv", index=False)
outliers.to_csv(summary_dir / "eda_outlier_candidates.csv", index=False)

print(f"Saved EDA summaries to: {summary_dir.resolve()}")

## Next steps

Potential follow-up analyses:
- stratify train/validation splits by `child_id` and `age_bucket` to reduce leakage
- compare transcript complexity and speaking rate by child/session
- integrate acoustic-level stats from audio files to tie text labels to signal properties